<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/Qwen-Image-2512T2I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ── Al inicio de la celda de setup, leer las keys ──────────────
import os
HF_TOKEN     = os.environ.get('HF_TOKEN', '')
TAILSCALE_KEY = os.environ.get('TAILSCALE_KEY', '')

if not HF_TOKEN:
    raise RuntimeError("❌ Falta HF_TOKEN — ejecuta la celda de config primero")
if not TAILSCALE_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY — ejecuta la celda de config primero")

# ── Reemplaza tu función download_hf por esta ──────────────────
def download_hf(url, dest_subdir, filename, token=None):
    dest_dir = os.path.join(MODELS, dest_subdir)
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        gb = os.path.getsize(dest) / 1024**3
        print(f"  ✅ {filename} ya existe ({gb:.2f} GB) — omitiendo")
        return
    print(f"  ↓  Descargando {filename} ...")
    header = f'--header="Authorization: Bearer {token}"' if token else ''
    ret = run(
        f'aria2c -c -x 8 -s 8 -k 5M --file-allocation=none '
        f'--console-log-level=warn {header} '
        f'"{url}" -d "{dest_dir}" -o "{filename}"',
        check=False
    )
    if ret != 0 or not os.path.exists(dest) or os.path.getsize(dest) < 1024 * 1024:
        print("  → aria2c falló, intentando con wget...")
        auth = f'--header="Authorization: Bearer {token}"' if token else ''
        run(f'wget -q --show-progress {auth} -O "{dest}" "{url}"', check=False)
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        gb = os.path.getsize(dest) / 1024**3
        print(f"  ✅ {filename} ({gb:.2f} GB)")
    else:
        print(f"  ❌ FALLO al descargar {filename}")

# ── Agrega esto al final de MODELS_TO_DOWNLOAD ─────────────────
MODELS_TO_DOWNLOAD.append((
    "https://huggingface.co/joanjlau/loraprueba0/resolve/main/YummyHDqwen.safetensors",
    "loras",
    "YummyHDqwen.safetensors",
))

# ── En el loop de descarga, pasa el token para el LoRA privado ──
for url, subfolder, filename in MODELS_TO_DOWNLOAD:
    token = HF_TOKEN if "joanjlau" in url else None
    download_hf(url, subfolder, filename, token=token)

  SETUP — Qwen-Image-2512 T2I · GGUF Q4_K_M + Lightning 4s BF16

[SYS] Instalando herramientas del sistema...
  $ apt-get update -qq && apt-get install -y -qq aria2 git wget curl
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

[0/5] ComfyUI base...
  ✅ ComfyUI ya existe
  $ git pull --ff-only
Already up to date.
  $ "/usr/bin/python3" -m pip install -r "/content/ComfyUI/requirements.txt" -q

[1/5] ComfyUI-GGUF (UnetLoaderGGUF)...
  ↻  ComfyUI-GGUF ya existe — actualizando
  $ git pull --ff-only
Already up to date.
  $ "/usr/bin/python3" -m pip install -r "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt" -q
  $ "/usr/bin/python3" -m pip install gguf -q --no-deps 2>/dev/null || "/usr/bin/python3" -m pip install gguf -q

[2/5] rgthree-comfy...
  ↻  rgthree-comfy ya existe — actualizando
  $ git pull --ff-only
Already up to date.
  $ "/us

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale (FIXED)

import os
import time

# --- CONFIGURACIÓN ---

TAILSCALE_AUTH_KEY = os.environ.get('TAILSCALE_KEY', '')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Ejecuta la celda de config primero")

# 1. Instalar Tailscale

!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar el daemon manualmente (sin systemd)

print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar que está corriendo

print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar a Tailscale

print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP

print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI

print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server --novram


Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:5 https://pkgs.tailscale.com/stable/ubuntu jammy InRelea

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 🔒 Lanzar ComfyUI con Cloudflare Tunnel

import os
import subprocess
import threading
import time

# --- CONFIGURACIÓN E INSTALACIÓN ---

print("📦 Instalando Cloudflared...")
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# --- FUNCIÓN PARA EL TÚNEL ---

def run_cloudflare():
    # Creamos el túnel apuntando al puerto 8188 (por defecto de ComfyUI)
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Buscamos la URL generada en los logs
    for line in p.stdout:
        if "trycloudflare.com" in line:
            print("\n" + "="*60)
            print("🌐 TU URL PÚBLICA ES:")
            print(line.strip().split(" ")[-1])
            print("="*60 + "\n")
        # Si quieres ver los logs de cloudflare, descomenta la siguiente línea:
        # print(line, end="")

# 1. Iniciar el túnel en segundo plano
threading.Thread(target=run_cloudflare, daemon=True).start()

# 2. Esperar un momento a que el túnel se establezca
time.sleep(5)

# 3. Lanzar ComfyUI
print("🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
# Usamos 127.0.0.1 porque el túnel de Cloudflare está escuchando localmente
!python main.py --dont-print-server